# Context A/B Tester

**See how the context you give an AI changes its answer.** Ask one question of one model two ways, side by side: **with vs without** your document, or **context A vs context B**. About a minute, free, no signup.

The classic aha: ask a question cold, then ask it again with your policy/doc pasted in — and watch the answer become correct. That is why grounding matters, shown in two plain calls.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.com).

---

> **Free tool, provided as-is, no warranty.** It uses *your own* API key and calls the provider directly. **Your key never leaves your machine / your Colab account and is never sent to pysolvr.** You are responsible for your own usage and costs. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2) — keys live in Colab Secrets or your shell environment, so a saved or shared copy contains no secrets.

## 1. Setup

Pulls the tool from the repo and installs the provider SDKs. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30
!wget -q -O context_ab_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/context-ab/context_ab_core.py
!wget -q -O prices.json         https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/context-ab/prices.json
print('ready')

## 2. Add your API key (do NOT type it into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab history, and any copy you share. Store it *outside* the notebook instead. The cell below only *reads* it.

### On Colab (recommended) — use the Secrets panel
1. Click the **key icon** (🔑 Secrets) in the left sidebar.
2. **Add a secret** for your model's provider, using the exact name:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for the secret. You only need the ONE provider whose model you use (this tool runs a single model).

Colab stores these encrypted, per-user — never in the notebook.

### On local Jupyter — environment variable
Set it in your shell *before* launching Jupyter: `export OPENAI_API_KEY=sk-...` then `jupyter lab`.

Then run the cell below. **Nothing is sent to pysolvr; your key stays on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. Contains no key itself.
import os

_KEY_NAMES = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY']
try:
    from google.colab import userdata  # Colab only
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set / notebook access off -- skip
except ImportError:
    pass  # not on Colab -- keys come from the shell environment

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
print('keys loaded for:', _loaded if _loaded else 'NONE -- add one via the Colab Secrets panel (key icon, left) with Notebook access ON, or set an env var. See cell above.')

## 3. Pick a model, your question, and your context

Use the provider's own model-ID string (the tool is model-agnostic). Examples: `gpt-5`, `claude-opus-5-5`, `gemini-4-argon`, `grok-4.7`. To find what exists and what it costs: **[llmprover.com/models](https://llmprover.com/models/)** and **[llmprover.com/api-pricing](https://llmprover.com/api-pricing/)**.

**Two modes** — set `MODE`:
- `'with-without'` (default): the same question answered **cold** vs **with `CONTEXT`**. Paste your document (a policy, FAQ, contract clause, ground-truth) into `CONTEXT`.
- `'a-b'`: the same question answered with **`CONTEXT_A`** vs **`CONTEXT_B`** (e.g. an old policy vs a new one). Fill both.

Paste your real text between the triple quotes.

In [ ]:
MODEL    = 'gpt-5'
QUESTION = 'Can I expense a 1,200 USD laptop without approval?'
MODE     = 'with-without'   # or 'a-b'

# For MODE = 'with-without': paste the document to ground the answer here.
CONTEXT = """
Expense policy: Any single purchase over 1,000 USD requires written manager approval in advance.
Purchases of 1,000 USD or less may be expensed without prior approval.
"""

# For MODE = 'a-b': paste the two contexts to compare (ignored in 'with-without' mode).
CONTEXT_A = """"""
CONTEXT_B = """"""

## 4. Run it

Both calls fire concurrently. You get each answer, token cost (from the dated price snapshot), and latency, side by side. **No verdict** — you read the two answers and judge for yourself whether the context changed the answer for the better.

In [ ]:
from context_ab_core import context_ab_core, format_result

result = context_ab_core(
    question=QUESTION,
    model=MODEL,
    mode=MODE,
    context=CONTEXT,
    context_a=CONTEXT_A,
    context_b=CONTEXT_B,
)
print(format_result(result))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.com/api-pricing](https://llmprover.com/api-pricing/).

If you're interested in more free tools to test and measure different aspects of an AI pipeline, see our [free AI tools](https://llmprover.com/tools/). For more advanced features — scored grounding across a whole document store, with a diagnostic and audit trail — see [Features](https://llmprover.com/features/).